# 03 - Feature Engineering
Create reusable features for Power BI and future ML models.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/cleaned/superstore_cleaned.csv")
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
df["Ship_Date"] = pd.to_datetime(df["Ship_Date"])

## Date Features

In [4]:
df["Order_Year"] = df["Order_Date"].dt.year
df["Order_Quarter"] = df["Order_Date"].dt.quarter
df["Order_Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Weekday"] = df["Order_Date"].dt.day_name()
df["Is_Weekend"] = df["Weekday"].isin(["Saturday","Sunday"])

## Shipping Features

In [6]:
df["Shipping_Days"] = (df["Ship_Date"]-df["Order_Date"]).dt.days
df["Shipping_Speed"] = pd.cut(
    df["Shipping_Days"],
    bins=[-1,2,5,100],
    labels=["Fast","Standard","Slow"]
)

## Financial Features

In [7]:
df["Profit Margin %"] = np.where(df["Sales"]!=0,
    (df["Profit"]/df["Sales"])*100,0)
df["Revenue Bucket"] = pd.qcut(df["Sales"],3,labels=["Low","Medium","High"])
df["Discount Applied"] = df["Discount"]>0

## Customer Features

In [9]:
cust = df.groupby("Customer_ID").agg(
    CustomerLifetimeRevenue=("Sales","sum"),
    CustomerLifetimeProfit=("Profit","sum"),
    TotalOrders=("Order_ID","nunique"),
    AvgOrderValue=("Sales","mean"),
    FirstPurchase=("Order_Date","min"),
    LastPurchase=("Order_Date","max")
).reset_index()

cust["Days Since Last Purchase"] = (
    cust["LastPurchase"].max()-cust["LastPurchase"]
).dt.days

df = df.merge(cust,on="Customer_ID",how="left")

## Product Features

In [11]:
prod = df.groupby("Product_ID").agg(
    ProductRevenue=("Sales","sum"),
    ProductProfit=("Profit","sum"),
    ProductQuantity=("Quantity","sum")
).reset_index()

df = df.merge(prod,on="Product_ID",how="left")

## RFM Features

In [13]:
snapshot = df["Order_Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Customer_ID").agg(
    Recency=("Order_Date", lambda x:(snapshot-x.max()).days),
    Frequency=("Order_ID","nunique"),
    Monetary=("Sales","sum")
).reset_index()

df = df.merge(rfm,on="Customer_ID",how="left")

## Export

In [15]:
df.to_csv("../data/cleaned/superstore_features.csv", index=False)
print("Exported feature engineered dataset.")

Exported feature engineered dataset.
